# 公式 o12 — MinDE System with Spatiocyte Simulator（空間・単分子）

> **出典（E-Cell4 公式）**: Examples / example09 — https://ecell4.e-cell.org/examples/example09.html
>
> o11 と同じ Min 極間振動を、今度は **Spatiocyte（格子ベースの単分子解像度）** で解く。1 分子ずつ格子上を拡散・反応させる。
>
> ⚠️ **本ノートは自動実行していません（出力なし）**。単分子解像度・`duration=240` 秒で**非常に重い**（実時間 数分〜十数分）＋
> 可視化はインタラクティブ。手元で実行してください。コードは公式のまま。

In [ ]:
# 公式コード（そのまま）。※単分子・非常に重い
from ecell4.prelude import *

with species_attributes():
    cytoplasm | {'radius': 1e-8, 'D': 0, 'dimension': 3}
    MinDatp | MinDadp | {'radius': 1e-8, 'D': 16e-12, 'location': 'cytoplasm', 'dimension': 3}
    MinEE_C | {'radius': 1e-8, 'D': 10e-12, 'location': 'cytoplasm', 'dimension': 3}
    membrane | {'radius': 1e-8, 'D': 0, 'location': 'cytoplasm', 'dimension': 2}
    MinD | MinEE_M | MinDEE | MinDEED | {'radius': 1e-8, 'D': 0.02e-12, 'location': 'membrane', 'dimension': 2}

with reaction_rules():
    membrane + MinDatp > MinD | 2.2e-8
    MinD + MinDatp > MinD + MinD | 3e-20
    MinD + MinEE_C > MinDEE | 5e-19
    MinDEE > MinEE_M + MinDadp | 1
    MinDadp > MinDatp | 5
    MinDEE + MinD > MinDEED | 5e-15
    MinDEED > MinDEE + MinDadp | 1
    MinEE_M > MinEE_C | 0.83

m = get_model()

f = spatiocyte.Factory(1e-8)                          # ボクセル半径 10 nm
w = f.world(Real3(4.6e-6, 1.1e-6, 1.1e-6))            # 桿菌サイズ (m)
w.bind_to(m)
rod = Rod(3.5e-6, 0.51e-6, w.edge_lengths() * 0.5)
w.add_structure(Species('cytoplasm'), rod)
w.add_structure(Species('membrane'), rod.surface())
w.add_molecules(Species('MinDadp'), 1300)
w.add_molecules(Species('MinDEE'), 700)

sim = f.simulator(w, m)
obs1 = FixedIntervalNumberObserver(0.1, ('MinDatp','MinDadp','MinEE_C','MinD','MinEE_M','MinDEE','MinDEED'))
obs2 = FixedIntervalHDF5Observer(1.0, 'minde%03d.h5')
duration = 240
sim.run(duration, (obs1, obs2))

## o11(meso) との違い

- **meso（o11）**: 空間を小区画に切り、区画ごとに分子**数**を扱う（速い・粗い）。
- **Spatiocyte（本ノート）**: 格子（ボクセル）上に**1 分子ずつ**配置し、単分子で拡散・反応（遅い・詳細。混雑や離散性を捉える）。

同じ Min 極間振動を、必要な解像度に応じてソルバを選べる、というのが E-Cell4 のマルチアルゴリズムの利点。

**要点**: `spatiocyte.Factory(voxel_radius)` ＋ `location`/`dimension`（膜=2D・細胞質=3D）で、
膜上拡散と細胞質拡散を単分子解像度で扱う。